# Train BEV v4 Cleaned

Self-contained notebook for `v4` training with:
- merged `train + val` pool;
- empty-GT removal;
- smart deduplication by near-static consecutive frames;
- test-matched validation split targeting about 200 samples;
- rover embeddings with `Other=0`, top-25 frequent train rovers getting unique IDs;
- stronger weight decay on the embedding table.

This notebook does not modify the original dataset structure. All caches and checkpoints are written only to `RUN_DIR`.


In [1]:
%load_ext autoreload
%autoreload 2

import os, copy, time, json, math, random, hashlib, zipfile
from pathlib import Path
from collections import Counter, defaultdict
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

DATA_TRAIN = Path('./autonomy_yandex_dataset_train/')
DATA_VAL   = Path('./autonomy_yandex_dataset_val/')
DATA_TEST  = Path('./autonomy_yandex_dataset_test/')

cfg = {
    'run_dir': './runs/v4_cleaned',
    'img_hw': (384, 704),
    'batch_size': 8,
    'grad_accum': 4,
    'epochs': 20,
    'warmup_epochs': 3,
    'lr_backbone': 3e-5,
    'lr_head': 3e-4,
    'weight_decay': 1e-4,
    'embed_weight_decay': 1e-2,
    'pos_weight': 5.0,
    'num_workers': 4,
    'seed': 42,
    'ema_decay': 0.995,
    'val_target_size': 200,
    'min_rover_count': 30,
    'topk_rovers': 25,
    'rover_emb_dim': 8,
    'rover_cond_dim': 8,
    'mae_dedup_thr': 0.02,
    'dedup_camera': '/camera/inner/frontal/middle',
    'use_clean_cache': True,
    'make_submission': False,
}

RUN_DIR = Path(cfg['run_dir'])
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = RUN_DIR / 'preproc_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

random.seed(cfg['seed'])
np.random.seed(cfg['seed'])
torch.manual_seed(cfg['seed'])

device = torch.device('mps')
print('device:', device)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

with open(RUN_DIR / 'config.json', 'w') as f:
    json.dump({k: str(v) for k, v in cfg.items()}, f, indent=2)


device: mps


In [2]:
CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)
Z_LEVELS = (0.3, 1.0, 2.0, 3.0)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def resolve_info_path(base_dir: Path, p) -> Path:
    p = Path(p)
    if p.is_absolute() and p.exists():
        return p
    if p.exists():
        return p
    cand = base_dir / p
    if cand.exists():
        return cand
    cand = base_dir.parent / p
    if cand.exists():
        return cand
    return base_dir.parent / p


def load_info_with_root(data_dir: Path, split_name: str) -> pd.DataFrame:
    df = pd.read_csv(data_dir / 'info.csv', index_col=0).reset_index(drop=True).copy()
    df['__data_root'] = str(data_dir)
    df['__source_split'] = split_name
    return df


def resolve_row_path(row: pd.Series, key: str) -> Path:
    return resolve_info_path(Path(row['__data_root']), row[key])


In [3]:
def compute_gt_stats(info_df: pd.DataFrame, cache_csv: Path | None = None) -> pd.DataFrame:
    if cache_csv is not None and cache_csv.exists():
        stats = pd.read_csv(cache_csv)
        if len(stats) == len(info_df):
            return stats

    rows = []
    for i, row in tqdm(info_df.iterrows(), total=len(info_df), desc='GT stats'):
        gt = np.load(resolve_row_path(row, GT_NAME)).squeeze()
        gt = np.where(gt < 0, 255, gt)
        valid = (gt != 255)
        pos = (gt == 1)
        rows.append({
            '__row_id': int(i),
            'coverage': float(valid.mean()),
            'valid_count': int(valid.sum()),
            'pos_count': int(pos.sum()),
        })
    stats = pd.DataFrame(rows)
    if cache_csv is not None:
        stats.to_csv(cache_csv, index=False)
    return stats


def build_img_hash(path: Path) -> np.ndarray:
    img = Image.open(path).convert('L').resize((32, 32), Image.BILINEAR)
    return np.asarray(img, dtype=np.float32) / 255.0


def smart_deduplicate(info_df: pd.DataFrame, mae_thr: float = 0.02,
                      camera_name: str = '/camera/inner/frontal/middle'):
    info_sorted = info_df.sort_values(['rover', 'ride_date', 'message_ts']).reset_index(drop=False)
    hash_cache = {}
    keep_row_ids = []
    duplicate_groups = []

    def get_hash_for_row(orig_row_idx: int, row: pd.Series):
        if orig_row_idx not in hash_cache:
            hash_cache[orig_row_idx] = build_img_hash(resolve_row_path(row, camera_name))
        return hash_cache[orig_row_idx]

    def flush_cluster(cluster):
        if not cluster:
            return
        if len(cluster) == 1:
            keep_row_ids.append(cluster[0]['orig_row_idx'])
            return
        best = sorted(
            cluster,
            key=lambda x: (-x['pos_count'], -x['valid_count'], x['message_ts'])
        )[0]
        keep_row_ids.append(best['orig_row_idx'])
        duplicate_groups.append({
            'kept_row_id': int(best['orig_row_idx']),
            'group_size': int(len(cluster)),
            'members': [int(x['orig_row_idx']) for x in cluster],
        })

    for (_, _), sub in tqdm(info_sorted.groupby(['rover', 'ride_date'], sort=False), desc='Smart dedup groups'):
        records = []
        for _, r in sub.iterrows():
            orig_idx = int(r['index'])
            records.append({
                'orig_row_idx': orig_idx,
                'row': info_df.iloc[orig_idx],
                'pos_count': int(info_df.iloc[orig_idx]['pos_count']),
                'valid_count': int(info_df.iloc[orig_idx]['valid_count']),
                'message_ts': str(info_df.iloc[orig_idx].get('message_ts', '')),
            })
        if not records:
            continue

        cluster = [records[0]]
        prev_hash = get_hash_for_row(records[0]['orig_row_idx'], records[0]['row'])
        for rec in records[1:]:
            cur_hash = get_hash_for_row(rec['orig_row_idx'], rec['row'])
            mae = float(np.mean(np.abs(cur_hash - prev_hash)))
            if mae < mae_thr:
                cluster.append(rec)
            else:
                flush_cluster(cluster)
                cluster = [rec]
            prev_hash = cur_hash
        flush_cluster(cluster)

    keep_row_ids = sorted(set(keep_row_ids))
    cleaned = info_df.iloc[keep_row_ids].reset_index(drop=True).copy()
    dup_df = pd.DataFrame(duplicate_groups)
    return cleaned, dup_df


def clean_merged_info(train_dir: Path, val_dir: Path, cache_dir: Path,
                      mae_thr: float = 0.02, dedup_camera: str = '/camera/inner/frontal/middle',
                      use_cache: bool = True):
    merged_cache = cache_dir / 'merged_cleaned.csv'
    stats_cache = cache_dir / 'merged_gt_stats.csv'
    dup_cache = cache_dir / 'dedup_report.csv'
    summary_path = cache_dir / 'clean_summary.json'

    if use_cache and merged_cache.exists() and summary_path.exists():
        info = pd.read_csv(merged_cache)
        dup_df = pd.read_csv(dup_cache) if dup_cache.exists() else pd.DataFrame()
        summary = json.loads(summary_path.read_text())
        return info, dup_df, summary

    train_info = load_info_with_root(train_dir, 'train')
    val_info = load_info_with_root(val_dir, 'val')
    merged = pd.concat([train_info, val_info], ignore_index=True)

    stats = compute_gt_stats(merged, cache_csv=stats_cache)
    merged = merged.join(stats[['coverage', 'valid_count', 'pos_count']])

    before = len(merged)
    merged = merged[merged['pos_count'] > 0].reset_index(drop=True).copy()
    after_empty = len(merged)

    deduped, dup_df = smart_deduplicate(
        merged,
        mae_thr=mae_thr,
        camera_name=dedup_camera,
    )
    after_dedup = len(deduped)

    deduped.to_csv(merged_cache, index=False)
    dup_df.to_csv(dup_cache, index=False)
    summary = {
        'merged_before_clean': int(before),
        'removed_empty_gt': int(before - after_empty),
        'after_empty_filter': int(after_empty),
        'removed_by_dedup': int(after_empty - after_dedup),
        'clean_total': int(after_dedup),
        'dedup_groups': int(len(dup_df)),
    }
    summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False))
    return deduped, dup_df, summary


In [4]:
def make_test_matched_split_target(info_df: pd.DataFrame,
                                   test_info_csv: Path,
                                   target_val_size: int = 200,
                                   group_cols=('rover', 'ride_date'),
                                   seed: int = 42,
                                   cache_path: Path | None = None):
    if cache_path is not None and cache_path.exists():
        d = np.load(cache_path)
        return d['train_idx'].tolist(), d['val_idx'].tolist()

    rng = np.random.RandomState(seed)
    info = info_df.reset_index(drop=True).copy()
    test_info = pd.read_csv(test_info_csv, index_col=0).reset_index(drop=True)
    test_counts = Counter(test_info['rover'])
    test_total = max(sum(test_counts.values()), 1)

    grouped = info.groupby(list(group_cols)).groups
    group_rows = []
    for key, idxs in grouped.items():
        rover = key[0]
        group_rows.append({
            'group_key': key,
            'rover': rover,
            'idxs': list(sorted(int(i) for i in idxs)),
            'size': int(len(idxs)),
        })
    groups_df = pd.DataFrame(group_rows)
    groups_df['test_weight'] = groups_df['rover'].map(lambda r: test_counts.get(r, 0) / test_total)
    groups_df = groups_df[groups_df['test_weight'] > 0].reset_index(drop=True)

    selected = set()

    def choose_groups(candidate_df: pd.DataFrame, target_count: int):
        candidate_df = candidate_df.sample(frac=1.0, random_state=rng.randint(0, 10**9)).reset_index(drop=True)
        chosen = []
        total = 0
        remaining = candidate_df.to_dict('records')
        while remaining and total < target_count:
            residual = target_count - total
            remaining.sort(key=lambda x: (abs(x['size'] - residual), x['size']))
            g = remaining.pop(0)
            chosen.append(g)
            total += g['size']
            if residual <= 0:
                break
        return chosen

    rover_order = [r for r, _ in test_counts.most_common() if r in set(groups_df['rover'])]
    selected_rows = []
    for rover in rover_order:
        rover_groups = groups_df[groups_df['rover'] == rover]
        if len(rover_groups) == 0:
            continue
        target = max(1, int(round(target_val_size * test_counts[rover] / test_total)))
        chosen = choose_groups(rover_groups, target)
        for g in chosen:
            if g['group_key'] not in selected:
                selected.add(g['group_key'])
                selected_rows.append(g)

    current = sum(g['size'] for g in selected_rows)
    remaining_df = groups_df[~groups_df['group_key'].isin(selected)].copy()
    while current < target_val_size and len(remaining_df) > 0:
        residual = target_val_size - current
        remaining_df = remaining_df.sample(frac=1.0, random_state=rng.randint(0, 10**9)).reset_index(drop=True)
        remaining_df = remaining_df.sort_values(['test_weight', 'size'], ascending=[False, True]).reset_index(drop=True)
        best_idx = min(range(len(remaining_df)), key=lambda i: (abs(int(remaining_df.iloc[i]['size']) - residual), -float(remaining_df.iloc[i]['test_weight'])))
        g = remaining_df.iloc[best_idx].to_dict()
        selected.add(g['group_key'])
        selected_rows.append(g)
        current += int(g['size'])
        remaining_df = remaining_df[remaining_df['group_key'] != g['group_key']].reset_index(drop=True)

    # Prune if we overshot too much.
    selected_df = pd.DataFrame(selected_rows)
    while len(selected_df) > 1 and selected_df['size'].sum() > target_val_size + 20:
        overflow = int(selected_df['size'].sum() - target_val_size)
        rover_group_counts = selected_df.groupby('rover').size().to_dict()
        candidate_ids = []
        for i, row in selected_df.iterrows():
            if rover_group_counts.get(row['rover'], 0) > 1:
                candidate_ids.append(i)
        if not candidate_ids:
            break
        drop_i = min(candidate_ids, key=lambda i: abs(int(selected_df.loc[i, 'size']) - overflow))
        selected_df = selected_df.drop(index=drop_i).reset_index(drop=True)

    val_groups = set(selected_df['group_key'].tolist())
    train_idx, val_idx = [], []
    for i, row in info.iterrows():
        key = tuple(row[c] for c in group_cols)
        if key in val_groups:
            val_idx.append(i)
        else:
            train_idx.append(i)

    if cache_path is not None:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez(cache_path, train_idx=np.array(train_idx), val_idx=np.array(val_idx))
    return train_idx, val_idx


def build_rover_vocab_from_train(train_df: pd.DataFrame,
                                 min_count: int = 30,
                                 topk: int = 25):
    counts = train_df['rover'].value_counts()
    eligible = counts[counts >= min_count]
    top = eligible.head(topk)
    vocab = {'__other__': 0}
    for i, rover in enumerate(top.index.tolist(), start=1):
        vocab[rover] = i
    stats_df = pd.DataFrame({
        'rover': counts.index,
        'count': counts.values,
        'embedding_id': [vocab.get(r, 0) for r in counts.index],
        'bucket': ['unique' if r in vocab and r != '__other__' else 'other' for r in counts.index],
    })
    return vocab, stats_df


def encode_rover(rover_name: str, rover_vocab: dict) -> int:
    return int(rover_vocab.get(rover_name, 0))


In [5]:
clean_info, dedup_report, clean_summary = clean_merged_info(
    DATA_TRAIN,
    DATA_VAL,
    cache_dir=CACHE_DIR,
    mae_thr=cfg['mae_dedup_thr'],
    dedup_camera=cfg['dedup_camera'],
    use_cache=cfg['use_clean_cache'],
)
print(json.dumps(clean_summary, indent=2, ensure_ascii=False))
display(clean_info.head(3))
if len(dedup_report):
    display(dedup_report.head(10))

train_idx, val_idx = make_test_matched_split_target(
    clean_info,
    DATA_TEST / 'info.csv',
    target_val_size=cfg['val_target_size'],
    seed=cfg['seed'],
    cache_path=CACHE_DIR / 'test_matched_val200_split.npz',
)
print('train_idx:', len(train_idx), 'val_idx:', len(val_idx))

train_info = clean_info.iloc[train_idx].reset_index(drop=True).copy()
val_info = clean_info.iloc[val_idx].reset_index(drop=True).copy()

rover_vocab, rover_stats = build_rover_vocab_from_train(
    train_info,
    min_count=cfg['min_rover_count'],
    topk=cfg['topk_rovers'],
)
rover_stats.to_csv(RUN_DIR / 'rover_embedding_stats.csv', index=False)
print('num rover classes including Other:', len(rover_vocab))
print('top rover mapping:', rover_vocab)
display(rover_stats.head(30))


Smart dedup groups: 100%|██████████| 1473/1473 [00:53<00:00, 27.57it/s]


{
  "merged_before_clean": 5000,
  "removed_empty_gt": 117,
  "after_empty_filter": 4883,
  "removed_by_dedup": 390,
  "clean_total": 4493,
  "dedup_groups": 173
}


,gt_occupancy_grid,header_ts,log_time,message_ts,/camera/inner/frontal/middle,/side/left/forward,/side/right/forward,/camera/inner/frontal/far,/camera/inner/frontal/middle/intrinsic_params,/camera/inner/frontal/middle/car_to_cam,...,predicted_occupancy_grid,ride_date,ride_time,rover,scene_part_order,__data_root,__source_split,coverage,valid_count,pos_count
0,autonomy_yandex_dataset_train/static_grids/163...,1633857774533809000,12:18:58,1633857774533809000,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,...,autonomy_yandex_dataset_train/predicted_static...,2021-10-10,12:18:58,orvy,0,autonomy_yandex_dataset_train,train,0.061972,1468,602
1,autonomy_yandex_dataset_train/static_grids/163...,1636812143899937000,15:34:56,1636812143899937000,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,...,autonomy_yandex_dataset_train/predicted_static...,2021-11-13,15:34:56,shelly,0,autonomy_yandex_dataset_train,train,0.227077,5379,3752
2,autonomy_yandex_dataset_train/static_grids/163...,1633600207233930000,12:28:29,1633600207233930000,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,...,autonomy_yandex_dataset_train/predicted_static...,2021-10-07,12:28:29,orvy,0,autonomy_yandex_dataset_train,train,0.225008,5330,1264


,kept_row_id,group_size,members
0,2271,2,"[2271, 3683]"
1,4101,2,"[4473, 4101]"
2,1366,3,"[306, 2710, 1366]"
3,98,5,"[1721, 2885, 209, 3630, 98]"
4,4151,2,"[4824, 4151]"
5,1703,3,"[526, 277, 1703]"
6,4867,3,"[4332, 4815, 4867]"
7,2549,2,"[3137, 2549]"
8,2112,2,"[2112, 3216]"
9,2330,2,"[2938, 2330]"


train_idx: 4273 val_idx: 220
num rover classes including Other: 26
top rover mapping: {'__other__': 0, 'orvy': 1, 'shelly': 2, 'lerita': 3, 'ward': 4, 'ravine': 5, 'greben': 6, 'lucky': 7, 'miro': 8, 'benzon': 9, 'natelio': 10, 'darron': 11, 'greton': 12, 'jurgie': 13, 'onipa': 14, 'targi': 15, 'kynde': 16, 'soan': 17, 'baland': 18, 'mika': 19, 'crozby': 20, 'litov': 21, 'hetera': 22, 'hankie': 23, 'stelard': 24, 'nikena': 25}


,rover,count,embedding_id,bucket
0,orvy,638,1,unique
1,shelly,496,2,unique
2,lerita,327,3,unique
3,ward,239,4,unique
4,ravine,208,5,unique
5,greben,194,6,unique
6,lucky,187,7,unique
7,miro,120,8,unique
8,benzon,114,9,unique
9,natelio,108,10,unique


In [6]:
class BEVDatasetV4Clean(Dataset):
    def __init__(self, info_df: pd.DataFrame, mode: str = 'train',
                 img_hw=(384, 704), aug: bool = False,
                 rover_vocab: dict | None = None):
        self.info = info_df.reset_index(drop=True).copy()
        self.mode = mode
        self.img_hw = img_hw
        self.aug = aug and (mode == 'train')
        self.rover_vocab = rover_vocab or {'__other__': 0}
        self.normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)

    def __len__(self):
        return len(self.info)

    def _resolve_path(self, row: pd.Series, key: str) -> Path:
        return resolve_info_path(Path(row['__data_root']), row[key])

    def _load_one_camera(self, row: pd.Series, cam_key: str, intr_key: str, c2c_key: str,
                         scale_aug: float = 1.0):
        img = Image.open(self._resolve_path(row, cam_key)).convert('RGB')
        intr_path = self._resolve_path(row, intr_key)
        car2cam_path = self._resolve_path(row, c2c_key)
        src_W, src_H = img.size
        H_t, W_t = self.img_hw

        s = min(W_t / src_W, H_t / src_H)
        new_W, new_H = int(round(src_W * s)), int(round(src_H * s))
        img_resized = img.resize((new_W, new_H), Image.BILINEAR)
        canvas = Image.new('RGB', (W_t, H_t), (0, 0, 0))
        pad_x = (W_t - new_W) // 2
        pad_y = (H_t - new_H) // 2
        canvas.paste(img_resized, (pad_x, pad_y))

        extra_s = 1.0
        extra_dx = 0
        extra_dy = 0
        if scale_aug > 1.0:
            sH, sW = int(round(H_t * scale_aug)), int(round(W_t * scale_aug))
            canvas = canvas.resize((sW, sH), Image.BILINEAR)
            extra_dx = random.randint(0, sW - W_t)
            extra_dy = random.randint(0, sH - H_t)
            canvas = canvas.crop((extra_dx, extra_dy, extra_dx + W_t, extra_dy + H_t))
            extra_s = scale_aug

        arr = np.array(canvas)
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        img_t = torch.from_numpy(arr).permute(2, 0, 1).float() / 255.0
        img_t = self.normalize(img_t)

        intr_full = np.load(intr_path)
        K = intr_full[:, :3].copy().astype(np.float32)
        K[0, 0] *= s; K[0, 2] *= s
        K[1, 1] *= s; K[1, 2] *= s
        K[0, 2] += pad_x
        K[1, 2] += pad_y
        K[0, 0] *= extra_s; K[0, 2] *= extra_s
        K[1, 1] *= extra_s; K[1, 2] *= extra_s
        K[0, 2] -= extra_dx
        K[1, 2] -= extra_dy

        car2cam = np.load(car2cam_path).astype(np.float32)
        return img_t, K, car2cam

    def _load_sample(self, idx: int):
        row = self.info.iloc[idx]
        scale_aug = random.uniform(1.0, 1.15) if self.aug else 1.0

        imgs, Ks, M = [], [], []
        for cam, intr, c2c in zip(CAMERA_NAMES, INTRINSICS_NAMES, CAR2CAM_NAMES):
            img_t, K, c = self._load_one_camera(row, cam, intr, c2c, scale_aug=scale_aug)
            imgs.append(img_t)
            Ks.append(torch.from_numpy(K))
            M.append(torch.from_numpy(c))

        out = {
            'images': torch.stack(imgs, dim=0),
            'intrinsics': torch.stack(Ks, dim=0),
            'car2cams': torch.stack(M, dim=0),
            'rover_id': torch.tensor(encode_rover(row.get('rover', '__other__'), self.rover_vocab), dtype=torch.long),
            'info_idx': idx,
        }
        if self.mode == 'test':
            return out

        gt = np.load(self._resolve_path(row, GT_NAME)).squeeze()
        gt = np.where(gt < 0, 255, gt).astype(np.int64)
        out['gt'] = torch.from_numpy(gt).unsqueeze(0)
        return out

    def __getitem__(self, idx: int):
        max_tries = 5
        last_err = None
        for k in range(max_tries):
            try_idx = (idx + k) % len(self.info)
            try:
                return self._load_sample(try_idx)
            except (OSError, ValueError, FileNotFoundError) as e:
                last_err = e
                continue
        raise RuntimeError(f'Failed to load sample after {max_tries} tries from idx={idx}: {last_err}')


In [ ]:
class _UNetBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class SmallUNet(nn.Module):
    def __init__(self, in_c, base_c=32, out_c=1):
        super().__init__()
        self.in_proj = nn.Conv2d(in_c, base_c, 1)
        c = [base_c, base_c * 2, base_c * 4, base_c * 8]
        self.enc1 = _UNetBlock(c[0], c[0])
        self.enc2 = _UNetBlock(c[0], c[1])
        self.enc3 = _UNetBlock(c[1], c[2])
        self.bot = _UNetBlock(c[2], c[3])
        self.dec3 = _UNetBlock(c[3] + c[2], c[2])
        self.dec2 = _UNetBlock(c[2] + c[1], c[1])
        self.dec1 = _UNetBlock(c[1] + c[0], c[0])
        self.out = nn.Conv2d(c[0], out_c, 1)
        self.pool = nn.MaxPool2d(2)

    def _up(self, x, ref):
        return F.interpolate(x, size=ref.shape[-2:], mode='bilinear', align_corners=False)

    def forward(self, x):
        x = self.in_proj(x)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bot(self.pool(e3))
        d3 = self.dec3(torch.cat([self._up(b, e3), e3], 1))
        d2 = self.dec2(torch.cat([self._up(d3, e2), e2], 1))
        d1 = self.dec1(torch.cat([self._up(d2, e1), e1], 1))
        return self.out(d1)


class _ResNet50Backbone(nn.Module):
    def __init__(self, pretrained: bool = True):
        super().__init__()
        weights = None
        if pretrained:
            try:
                weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
            except Exception:
                weights = None
        rn = torchvision.models.resnet50(weights=weights)
        self.stem = nn.Sequential(rn.conv1, rn.bn1, rn.relu, rn.maxpool)
        self.layer1 = rn.layer1
        self.layer2 = rn.layer2
        self.proj = nn.Conv2d(512, 128, 1)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.proj(x)
        return x


class MultiCamBEVv4Clean(nn.Module):
    def __init__(self, num_rover_classes: int,
                 rover_emb_dim: int = 8,
                 rover_cond_dim: int = 8,
                 n_cameras: int = 4,
                 freeze_backbone: bool = False):
        super().__init__()
        self.n_cameras = n_cameras
        self.bev_h, self.bev_w = BEV_H, BEV_W
        self.x_range, self.y_range = X_RANGE, Y_RANGE
        self.z_levels = Z_LEVELS
        self.rover_cond_dim = rover_cond_dim

        self.backbone = _ResNet50Backbone(pretrained=True)
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.feat_proj = nn.Conv2d(128, 64, 1)
        self.register_buffer('ego_voxels', self._make_ego_voxels(), persistent=False)

        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        nn.init.normal_(self.rover_embed.weight, std=0.02)
        self.rover_mlp = nn.Sequential(
            nn.Linear(rover_emb_dim, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, rover_cond_dim),
            nn.ReLU(inplace=True),
        )

        in_c = 64 * len(self.z_levels) + rover_cond_dim
        self.bev_decoder = SmallUNet(in_c=in_c, base_c=32, out_c=1)

    def _make_ego_voxels(self):
        xs = torch.linspace(self.x_range[0] + BEV_RES / 2, self.x_range[1] - BEV_RES / 2, self.bev_h)
        ys = torch.linspace(self.y_range[0] + BEV_RES / 2, self.y_range[1] - BEV_RES / 2, self.bev_w)
        zs = torch.tensor(self.z_levels, dtype=torch.float32)
        Z, X, Y = torch.meshgrid(zs, xs, ys, indexing='ij')
        ones = torch.ones_like(X)
        return torch.stack([X, Y, Z, ones], dim=-1)

    def forward(self, images, intrinsics, car2cams, rover_ids):
        B, N, C_, Hi, Wi = images.shape
        assert N == self.n_cameras

        feat = self.backbone(images.reshape(B * N, C_, Hi, Wi))
        feat = self.feat_proj(feat)
        Hf, Wf = feat.shape[-2:]
        feat = feat.reshape(B, N, 64, Hf, Wf)

        Z, H, W, _ = self.ego_voxels.shape
        V = Z * H * W
        voxels = self.ego_voxels.reshape(-1, 4).unsqueeze(0).unsqueeze(0).expand(B, N, -1, -1)

        p_cam = torch.einsum('bnij,bnvj->bniv', car2cams, voxels)
        p_cam_3d = p_cam[:, :, :3]
        uv_homog = torch.einsum('bnij,bnjv->bniv', intrinsics, p_cam_3d)
        z = uv_homog[:, :, 2]
        u = uv_homog[:, :, 0] / z.clamp(min=1e-3)
        v = uv_homog[:, :, 1] / z.clamp(min=1e-3)

        u_n = 2.0 * u / Wi - 1.0
        v_n = 2.0 * v / Hi - 1.0
        valid = (z > 0.1) & (u_n.abs() <= 1.0) & (v_n.abs() <= 1.0)

        grid = torch.stack([u_n, v_n], dim=-1).reshape(B * N, V, 1, 2)
        sampled = F.grid_sample(
            feat.reshape(B * N, 64, Hf, Wf),
            grid,
            mode='bilinear',
            padding_mode='zeros',
            align_corners=False,
        )
        sampled = sampled.squeeze(-1).reshape(B, N, 64, V)

        valid_f = valid.float().unsqueeze(2)
        sampled = sampled * valid_f
        agg = sampled.sum(dim=1) / valid_f.sum(dim=1).clamp(min=1.0)
        agg = agg.reshape(B, 64, Z, H, W).reshape(B, 64 * Z, H, W)

        rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
        rover_map = rover_feat.expand(-1, -1, H, W)
        agg = torch.cat([agg, rover_map], dim=1)
        return self.bev_decoder(agg)


In [ ]:
def _lovasz_grad(gt_sorted: torch.Tensor) -> torch.Tensor:
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1.0 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard


def lovasz_hinge_flat(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    if labels.numel() == 0:
        return logits.sum() * 0.0
    signs = 2.0 * labels.float() - 1.0
    errors = 1.0 - logits * signs
    errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
    gt_sorted = labels[perm]
    grad = _lovasz_grad(gt_sorted)
    return torch.dot(F.relu(errors_sorted), grad)


class CompoundLossV2(nn.Module):
    def __init__(self, pos_weight: float = 5.0,
                 weight_bce: float = 0.5,
                 weight_dice: float = 0.3,
                 weight_lovasz: float = 0.2,
                 ignore_value: int = 255):
        super().__init__()
        self.register_buffer('pos_weight', torch.tensor([pos_weight]))
        self.weight_bce = weight_bce
        self.weight_dice = weight_dice
        self.weight_lovasz = weight_lovasz
        self.ignore_value = ignore_value

    def forward(self, logits: torch.Tensor, gt: torch.Tensor):
        valid = (gt != self.ignore_value)
        gt_f = (gt == 1).float()

        bce_per = F.binary_cross_entropy_with_logits(logits, gt_f, pos_weight=self.pos_weight, reduction='none')
        bce = (bce_per * valid.float()).sum() / valid.float().sum().clamp(min=1.0)

        prob = torch.sigmoid(logits) * valid.float()
        gt_d = gt_f * valid.float()
        inter = (prob * gt_d).sum(dim=(1, 2, 3))
        denom = prob.sum(dim=(1, 2, 3)) + gt_d.sum(dim=(1, 2, 3))
        dice = (1.0 - (2 * inter + 1) / (denom + 1)).mean()

        lov_logits = logits[valid]
        lov_gt = gt_f[valid]
        lov = lovasz_hinge_flat(lov_logits, lov_gt) if lov_logits.numel() > 0 else logits.sum() * 0.0

        total = self.weight_bce * bce + self.weight_dice * dice + self.weight_lovasz * lov
        parts = {'bce': bce.item(), 'dice': dice.item(), 'lovasz': lov.item()}
        return total, parts


@torch.no_grad()
def iou_binary_batch(logits: torch.Tensor, gt: torch.Tensor, threshold: float = 0.5, ignore_value: int = 255):
    probs = torch.sigmoid(logits)
    pred = probs > threshold
    valid = gt != ignore_value
    gt_b = (gt == 1) & valid
    pred = pred & valid
    inter = (pred & gt_b).sum().item()
    union = (pred | gt_b).sum().item()
    return inter, union


@torch.no_grad()
def evaluate_iou(model, loader, threshold=0.5):
    model.eval()
    inter = 0
    union = 0
    for batch in loader:
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(images, intr, c2c, rover_id).float()
        i, u = iou_binary_batch(logits, gt, threshold=threshold)
        inter += i
        union += u
    return inter / max(union, 1)


@torch.no_grad()
def streaming_threshold_sweep(model, loader, thresholds):
    model.eval()
    inter = torch.zeros(len(thresholds), dtype=torch.float64)
    union = torch.zeros(len(thresholds), dtype=torch.float64)
    for batch in tqdm(loader, desc='threshold sweep'):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt']
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(images, intr, c2c, rover_id).float()
        probs = torch.sigmoid(logits).cpu()
        valid = gt != 255
        gt_b = ((gt == 1) & valid).float()
        for ti, t in enumerate(thresholds):
            pred = ((probs > t) & valid).float()
            inter[ti] += (pred * gt_b).sum().item()
            union[ti] += ((pred + gt_b).clamp(0, 1)).sum().item()
    return {float(t): float(inter[i] / max(union[i].item(), 1.0)) for i, t in enumerate(thresholds)}


In [ ]:
ds_train_warm = BEVDatasetV4Clean(train_info, mode='train', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)
ds_train_aug = BEVDatasetV4Clean(train_info, mode='train', img_hw=cfg['img_hw'], aug=True, rover_vocab=rover_vocab)
ds_val = BEVDatasetV4Clean(val_info, mode='val', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)

loader_warm = DataLoader(ds_train_warm, batch_size=cfg['batch_size'], shuffle=True,
                         num_workers=cfg['num_workers'], pin_memory=(device.type == 'cuda'), drop_last=True)
loader_train = DataLoader(ds_train_aug, batch_size=cfg['batch_size'], shuffle=True,
                          num_workers=cfg['num_workers'], pin_memory=(device.type == 'cuda'), drop_last=True)
loader_val = DataLoader(ds_val, batch_size=cfg['batch_size'], shuffle=False,
                        num_workers=cfg['num_workers'], pin_memory=(device.type == 'cuda'))

sample = ds_train_warm[0]
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(k, tuple(v.shape), v.dtype)
    else:
        print(k, type(v), v)


In [ ]:
model = MultiCamBEVv4Clean(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
).to(device)

backbone_params = []
embed_params = []
other_params = []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if name.startswith('backbone.'):
        backbone_params.append(p)
    elif name.startswith('rover_embed.'):
        embed_params.append(p)
    else:
        other_params.append(p)

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': cfg['lr_backbone'], 'weight_decay': cfg['weight_decay']},
    {'params': other_params, 'lr': cfg['lr_head'], 'weight_decay': cfg['weight_decay']},
    {'params': embed_params, 'lr': cfg['lr_head'], 'weight_decay': cfg['embed_weight_decay']},
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'], eta_min=1e-6)
loss_fn = CompoundLossV2(pos_weight=cfg['pos_weight']).to(device)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

ema_model = copy.deepcopy(model).eval()
for p in ema_model.parameters():
    p.requires_grad = False

@torch.no_grad()
def update_ema(ema_model, model, decay):
    ema_params = dict(ema_model.named_parameters())
    model_params = dict(model.named_parameters())
    for name, param in model_params.items():
        ema_params[name].mul_(decay).add_(param.data, alpha=1.0 - decay)
    ema_buffers = dict(ema_model.named_buffers())
    model_buffers = dict(model.named_buffers())
    for name, buf in model_buffers.items():
        ema_buffers[name].copy_(buf)

print('params M:', sum(p.numel() for p in model.parameters()) / 1e6)


In [ ]:
thresholds = [round(x, 2) for x in np.arange(0.30, 0.81, 0.02)]
log = []
best_iou = -1.0
best_ema_iou = -1.0
start = time.time()

for epoch in range(cfg['epochs']):
    model.train()
    loader = loader_warm if epoch < cfg['warmup_epochs'] else loader_train
    phase = 'WARM' if epoch < cfg['warmup_epochs'] else 'AUG'

    losses = []
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc=f'ep{epoch:02d} [{phase}]')
    for step, batch in enumerate(pbar):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(images, intr, c2c, rover_id)
            loss, parts = loss_fn(logits, gt)
            loss = loss / cfg['grad_accum']

        scaler.scale(loss).backward()
        if (step + 1) % cfg['grad_accum'] == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            update_ema(ema_model, model, cfg['ema_decay'])

        losses.append(loss.item() * cfg['grad_accum'])
        if step % 20 == 0:
            pbar.set_postfix(loss=f'{np.mean(losses[-50:]):.3f}', bce=f"{parts['bce']:.2f}")

    scheduler.step()

    val_iou_05 = evaluate_iou(model, loader_val, threshold=0.5)
    val_iou_05_ema = evaluate_iou(ema_model, loader_val, threshold=0.5)
    elapsed = (time.time() - start) / 60

    row = {
        'epoch': epoch,
        'loss': float(np.mean(losses)),
        'val_iou_05': float(val_iou_05),
        'val_iou_05_ema': float(val_iou_05_ema),
        'minutes': float(elapsed),
    }

    if epoch % 2 == 0 or epoch == cfg['epochs'] - 1:
        sweep_model = streaming_threshold_sweep(model, loader_val, thresholds)
        best_t_model, best_iou_model = max(sweep_model.items(), key=lambda kv: kv[1])
        sweep_ema = streaming_threshold_sweep(ema_model, loader_val, thresholds)
        best_t_ema, best_iou_ema = max(sweep_ema.items(), key=lambda kv: kv[1])
        row.update({
            'best_t_model': float(best_t_model),
            'best_iou_model': float(best_iou_model),
            'best_t_ema': float(best_t_ema),
            'best_iou_ema': float(best_iou_ema),
        })
        print(f"ep{epoch:02d} | loss {np.mean(losses):.3f} | val@0.5 {val_iou_05:.4f} | ema@0.5 {val_iou_05_ema:.4f} | best_t {best_t_model:.2f}/{best_t_ema:.2f} | best_iou {best_iou_model:.4f}/{best_iou_ema:.4f} | {elapsed:.1f}m")

        if best_iou_model > best_iou:
            best_iou = best_iou_model
            torch.save({
                'model_type': 'v4_cleaned',
                'model': model.state_dict(),
                'epoch': epoch,
                'best_iou': best_iou_model,
                'best_t': best_t_model,
                'rover_vocab': rover_vocab,
                'cfg': cfg,
            }, RUN_DIR / 'best.pt')
            print('  new best model:', best_iou_model)

        if best_iou_ema > best_ema_iou:
            best_ema_iou = best_iou_ema
            torch.save({
                'model_type': 'v4_cleaned',
                'model': model.state_dict(),
                'ema': ema_model.state_dict(),
                'epoch': epoch,
                'best_ema_iou': best_iou_ema,
                'best_t_ema': best_t_ema,
                'rover_vocab': rover_vocab,
                'cfg': cfg,
            }, RUN_DIR / 'ema_best.pt')
            print('  new best ema:', best_iou_ema)
    else:
        print(f"ep{epoch:02d} | loss {np.mean(losses):.3f} | val@0.5 {val_iou_05:.4f} | ema@0.5 {val_iou_05_ema:.4f} | {elapsed:.1f}m")

    log.append(row)
    pd.DataFrame(log).to_csv(RUN_DIR / 'log.csv', index=False)
    torch.save({
        'model_type': 'v4_cleaned',
        'model': model.state_dict(),
        'ema': ema_model.state_dict(),
        'epoch': epoch,
        'rover_vocab': rover_vocab,
        'cfg': cfg,
    }, RUN_DIR / 'last.pt')


In [ ]:
ckpt = torch.load(RUN_DIR / 'ema_best.pt', map_location=device) if (RUN_DIR / 'ema_best.pt').exists() else torch.load(RUN_DIR / 'best.pt', map_location=device)
model_eval = MultiCamBEVv4Clean(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
).to(device)
state = ckpt['ema'] if 'ema' in ckpt else ckpt['model']
model_eval.load_state_dict(state, strict=False)

thresholds = [round(x, 2) for x in np.arange(0.20, 0.81, 0.02)]
iou_by_t = streaming_threshold_sweep(model_eval, loader_val, thresholds)
best_t, best_iou = max(iou_by_t.items(), key=lambda kv: kv[1])
print('best threshold:', best_t, 'best_iou:', best_iou)
for t, iou in iou_by_t.items():
    marker = ' <- best' if t == best_t else ''
    print(f't={t:.2f}  iou={iou:.4f}{marker}')


## Notes

- `clean_merged_info(...)` only writes caches and reports into `RUN_DIR / preproc_cache`.
- The original dataset folders are never modified.
- The `rover_vocab` is built strictly from the cleaned training split, so rare rovers and unseen rovers map to `Other=0`.
- Validation is selected from the merged cleaned pool and targets about 200 samples while staying group-aware by `(rover, ride_date)`.
